# Retail Sales Forecasting — Exploratory Data Analysis

Exploring the Walmart Store Sales dataset to understand seasonality, trends, and store-level patterns before building forecasting models.

**Dataset:** [Kaggle — Walmart Recruiting Store Sales Forecasting](https://www.kaggle.com/c/walmart-recruiting-store-sales-forecasting)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

# Load data
train = pd.read_csv("../data/train.csv")
stores = pd.read_csv("../data/stores.csv")
features = pd.read_csv("../data/features.csv")

# Merge
df = train.merge(stores, on="Store").merge(features, on=["Store", "Date"], how="left")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"Stores: {df['Store'].nunique()}, Departments: {df['Dept'].nunique()}")
df.head()

## 1. Overall Sales Trend

In [ ]:
weekly_total = df.groupby("Date")["Weekly_Sales"].sum().reset_index()

plt.figure(figsize=(14, 5))
plt.plot(weekly_total["Date"], weekly_total["Weekly_Sales"] / 1e6, linewidth=1.5)
plt.title("Total Weekly Sales Across All Stores")
plt.ylabel("Sales (millions $)")
plt.xlabel("Date")
plt.tight_layout()
plt.show()

## 2. Seasonality Analysis

In [ ]:
df["month"] = df["Date"].dt.month
df["week_of_year"] = df["Date"].dt.isocalendar().week.astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Monthly pattern
monthly = df.groupby("month")["Weekly_Sales"].mean()
axes[0].bar(monthly.index, monthly.values, color="steelblue")
axes[0].set_title("Average Weekly Sales by Month")
axes[0].set_xlabel("Month")

# Holiday impact
holiday_sales = df.groupby("IsHoliday")["Weekly_Sales"].mean()
axes[1].bar(["Non-Holiday", "Holiday"], holiday_sales.values, color=["steelblue", "salmon"])
axes[1].set_title("Average Sales: Holiday vs Non-Holiday Weeks")

plt.tight_layout()
plt.show()

print(f"Holiday uplift: {(holiday_sales[True] / holiday_sales[False] - 1) * 100:.1f}%")

## 3. Store-Level Variation

In [ ]:
store_sales = df.groupby("Store")["Weekly_Sales"].agg(["mean", "std"]).sort_values("mean", ascending=False)

plt.figure(figsize=(14, 5))
plt.bar(range(len(store_sales)), store_sales["mean"], color="steelblue", alpha=0.7)
plt.errorbar(range(len(store_sales)), store_sales["mean"], yerr=store_sales["std"],
             fmt="none", color="gray", alpha=0.5)
plt.title("Average Weekly Sales by Store (with std dev)")
plt.xlabel("Store (ranked)")
plt.ylabel("Weekly Sales ($)")
plt.tight_layout()
plt.show()

## 4. Key Takeaways\n",

1. **Strong seasonality**: Clear December spike (holiday shopping) and dip in January\n",
2. **Holiday weeks**: ~7-10% higher average sales during holiday weeks\n",
3. **Store heterogeneity**: Large variation across stores — models should account for store-level effects\n",
4. **Feature engineering priorities**: Calendar features, lag features (same week last year), and rolling averages will be important